In [36]:
def section_aware_split(text: str, max_chunk_len: int = 1500) -> list:
    """
    Chunk a Markdown-style document into hierarchical sections (using #, ##, ###) 
    and return structured chunks with section path and level.
    """
    import re

    lines = text.splitlines()
    chunks = []
    current_chunk_lines = []
    current_path = []

    def flush_chunk():
        if not current_chunk_lines:
            return
        content = "\n".join(current_chunk_lines).strip()
        if content:
            chunks.append({
                "section_path": current_path.copy(),
                "level": len(current_path),
                "content": content
            })

    for line in lines:
        header_match = re.match(r"^(#{1,6})\s+(.*)", line)
        if header_match:
            # New header found
            flush_chunk()
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_path = current_path[:level - 1] + [title]
            current_chunk_lines = [line]
        else:
            current_chunk_lines.append(line)

    flush_chunk()
    return chunks

In [37]:
import os
import nest_asyncio

from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.vector_stores import VectorStoreQueryResult
from qdrant_client import QdrantClient, AsyncQdrantClient
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings
from typing import List
from dotenv import load_dotenv
import json
import os
nest_asyncio.apply()
load_dotenv(dotenv_path=".env.dev")
# Apply the monkey patch
from chainlit_app.patches import patch
patch.apply_patch()

2025-08-06 13:19:50,299 - INFO - Patched TextEmbeddingsInference._call_api with custom synchronous API handling
2025-08-06 13:19:50,299 - INFO - Patched TextEmbeddingsInference._acall_api with custom asynchronous API handling


In [38]:
from docx import Document

def extract_tables_as_markdown(docx_path):
    doc = Document(docx_path)
    markdown_tables = []
    for table in doc.tables:
        rows = []
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells]
            rows.append("| " + " | ".join(cells) + " |")
        if rows:
            header = rows[0]
            separator = "| " + " | ".join(["---"] * len(table.columns)) + " |"
            markdown_table = "\n".join([header, separator] + rows[1:])
            markdown_tables.append(markdown_table)
    return markdown_tables

In [39]:
from llama_index.readers.file import DocxReader
from llama_index.core.schema import Document
import os
import glob
import re

# ——————————————
# CONFIGURATION
# ——————————————
DOCX_FOLDER = "documents/"
SHAREPOINT_BASE_URL = "https://cpaxtra.sharepoint.com/sites/forms-library"

# ——————————————
# SECTION TITLE HELPER
# ——————————————
def extract_section_title(chunk: str) -> str:
    """
    Extract section title from chunk marked as [SECTION] ... or fallback to first line.
    """
    match = re.search(r"\[SECTION\] (.*?)\n", chunk)
    if match:
        return match.group(1).strip()
    # fallback to first non-empty line
    lines = [line.strip() for line in chunk.splitlines() if line.strip()]
    return lines[0] if lines else "unknown"

# ——————————————
# STEP 1: Discover all .docx files
# ——————————————
all_paths = glob.glob(os.path.join(DOCX_FOLDER, "*.docx"))
print(f"Found {len(all_paths)} .docx file(s):")
for p in all_paths:
    print("  •", p)

# ——————————————
# STEP 2: Load each DOCX and wrap as Document
# ——————————————
reader = DocxReader()
raw_documents = []
for file_path in all_paths:
    docx_pages = reader.load_data(file_path)
    for page_obj in docx_pages:
        raw_documents.append(
            Document(
                text=page_obj.text,
                metadata={"source": os.path.basename(file_path)}
            )
        )

print(f"Loaded {len(raw_documents)} raw Document(s) from all .docx files.")

# ——————————————
# STEP 3: Chunk each Document semantically with metadata
# ——————————————
nodes = []
for doc in raw_documents:
    file_name = doc.metadata.get("source", "")
    attachment_link = f"{SHAREPOINT_BASE_URL}/{file_name}"
    section_chunks = section_aware_split(doc.text)
    
    for i, chunk in enumerate(section_chunks):
        nodes.append(
            Document(
                text=chunk["content"],
                metadata={
                    **doc.metadata,
                    "chunk_id": i,
                    "section_path": chunk["section_path"],
                    "level": chunk["level"],
                    "attachment_link": f"{SHAREPOINT_BASE_URL}/{file_name}"
                }
            )
        )

print(f"After splitting, we have {len(nodes)} chunked Documents (nodes).")

# ——————————————
# FINAL: Assign to `documents` so rest of pipeline stays unchanged
# ——————————————
documents = nodes

Found 15 .docx file(s):
  • documents/FAQ_Narrative_3 Non Trade Supplier.docx
  • documents/การเพิ่มและแก้ไขข้อมูลคู่ค้าNon-trade.docx
  • documents/อำนาจอนุมัติรายจ่ายทั่วไป.docx
  • documents/อำนาจอนุมัติรายจ่ายสำหรับ Purchase Requisition.docx
  • documents/อำนาจอนุมัติ DoA และ LoA.docx
  • documents/FAQ_Narrative_2 Trade Supplier.docx
  • documents/เงินลงทุนในโครงการ.docx
  • documents/FAQ_Narrative - Org Structure and General Question.docx
  • documents/การเบิกค่าใช้จ่ายพนักงาน.docx
  • documents/การเพิ่มข้อมูลคู่ค้า และการจ่ายเงิน (Trade).docx
  • documents/การเพิ่ม คัดเลือกลูกค้า การต่อสัญญา และการติดตามหนี้.docx
  • documents/FAQ_Narrative_7 Asset.docx
  • documents/การบริหารสินเชื่อสำหรับธุรกิจ B2B.docx
  • documents/FAQ_Narrative_1 DoA LOA_Proj App_Payment App.docx
  • documents/FAQ_Narrative_4 Mall.docx
Loaded 15 raw Document(s) from all .docx files.
After splitting, we have 475 chunked Documents (nodes).


In [40]:
from llama_index.core import StorageContext

## Setup BGE-M3 Embedding service

In [41]:
# BGE-M3 Embedding Configuration using environment variables
embed_model = TextEmbeddingsInference(
    model_name=os.getenv("EMBED_MODEL_ID"),
    base_url=os.getenv("EMBED_BASE_URL"),
    auth_token=f"Bearer {os.getenv('API_KEY_CHATBOT')}",
    timeout=60,
    embed_batch_size=10,
)

Settings.embed_model = embed_model
Settings.chunk_size = 1024

print(f"🔑 Using BGE-M3 Embedding Service")
print(f"🔢 Using Model: {os.getenv('EMBED_MODEL_ID')}")
print(f"🌐 Using Base URL: {os.getenv('EMBED_BASE_URL')}")

🔑 Using BGE-M3 Embedding Service
🔢 Using Model: BAAI/bge-m3
🌐 Using Base URL: https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3


## Innitiates VectorStore database (Qdrant)

In [42]:
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore
import os

# Initialize Qdrant client with HTTP (not gRPC) - matching working config
client = QdrantClient(
    url="http://localhost:6433",  # Using HTTP endpoint exposed by Docker
    api_key=os.getenv("QDRANT_API_KEY"),
    prefer_grpc=False,            # Disable gRPC to avoid connection issues
    timeout=60,
    check_compatibility=False     # Suppress version mismatch warning
)

# Load collection name from environment
collection_name = os.getenv("QDRANT_COLLECTION_NAME")

# Delete collection if it exists
if client.collection_exists(collection_name):
    print(f"Collection '{collection_name}' already exists.")
    print(f"Deleting existing collection: {collection_name}")
    client.delete_collection(collection_name)

# Create Qdrant vector store with hybrid search enabled
vector_store = QdrantVectorStore(
    collection_name=collection_name,
    client=client,
    enable_hybrid=True,
    batch_size=20,
    prefer_grpc=False             # Match client setting
)

/tmp/ipykernel_915774/1517703558.py:6: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(
2025-08-06 13:19:50,939 - INFO - HTTP Request: GET http://localhost:6433/collections/FAQ_DATA/exists "HTTP/1.1 200 OK"
2025-08-06 13:19:50,948 - INFO - HTTP Request: DELETE http://localhost:6433/collections/FAQ_DATA "HTTP/1.1 200 OK"
2025-08-06 13:19:50,950 - INFO - HTTP Request: GET http://localhost:6433/collections/FAQ_DATA/exists "HTTP/1.1 200 OK"
2025-08-06 13:19:50,953 - INFO - HTTP Request: GET http://localhost:6433/collections/FAQ_DATA/exists "HTTP/1.1 200 OK"


Collection 'FAQ_DATA' already exists.
Deleting existing collection: FAQ_DATA


2025-08-06 13:19:53,395 - INFO - HTTP Request: GET http://localhost:6433/collections/FAQ_DATA/exists "HTTP/1.1 200 OK"


## Start embedding process.... into vector database

In [43]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents=documents, embed_model=embed_model, storage_context=storage_context,
)

2025-08-06 13:19:56,851 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:56,947 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:57,043 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:57,115 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:57,192 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:57,294 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:57,361 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:19:57,420 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "H

## Try to retrive relavent nodes with question.

In [44]:
# Use the same embedding model for retrieval (no need to recreate)
search_query_retriever = index.as_retriever()

search_query_retrieved_nodes = search_query_retriever.retrieve(
"Do all Walmart locations offer scan & go?"
)

2025-08-06 13:20:22,378 - INFO - HTTP Request: POST https://api-cpxis.lotuss.com/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-08-06 13:20:22,387 - INFO - HTTP Request: POST http://localhost:6433/collections/FAQ_DATA/points/search/batch "HTTP/1.1 200 OK"


In [45]:
from llama_index.core.response.notebook_utils import display_source_node
for n in search_query_retrieved_nodes:
    display_source_node(n, source_length=2000)

**Node ID:** f857dd9d-7505-4c96-a153-669e40ffb55a<br>**Similarity:** 0.45676488<br>**Text:** #ค่าบริการเปลียนชื่อคู่สัญญา เก็บที่ 10,000[รวมvat] หรือ 10,000[ยังไม่รวมvat]	

- ขึ้นอยู่กับ Leasing Manager ตกลงกับผู้เช่า ได้ทั้ง 2 แบบ



#กรณีจ่ายเงินประกันแล้วแต่ไม่ได้เข้าพื้นที่จริง เนื่องจากปัญหาจากทางผู้ให้เช่า[lotus's] เช่น แผนผังหน้างานร้านไม่เป็นไปตามข้อตกลง   ลูกค้าสามารถขอคืน หรือย้ายเงินประกันไปสาขาอื่น หรือ ใช้จ่ายค่าเช่าสาขาอื่น ได้หรือไม่ และใช้เอกสารหลักฐานใดในการขอคืน หรือ ย้ายไปใช้เป้นเงินประกันสาขาอื่น หรือ จ่ายค่าเช่าสาขาอื่นภายใต้่ชื่อลูกค้าเดียวกันมีอยู่กับLotus's ได้ / เอกกสาร 

1. Memo ชี้แจงสาเหตุและอนุมัติใช้เงินประกันที่ได้รับอนุมัติจากWL3 Leasing 

2.เอกสารประกอบอื่นๆ เช่น Plan renovate หรือ ใบเสนอราคาที่มีแผนผังชัดเจน



#ช่วงร้านค้าเข้าตกแต่งร้าน หรือก่อสร้างร้าน มีการเก็บเงินประกันหรือไม่ ถ้ามี เก็บอย่างไร	

- ผู้รับเหมาจ่ายเป็นแคชเชียร์เช็คให้กับ constaction team และจะคืนแคชเชียร์เช็คฉบับนั้นให้กับผู้รับเหมาเมื่อสิ้นสุดโครงการ โดยไม่มีการนำแคชเชียร์เช็คไปขึ้นเงินเข้าบัญชี<br>

**Node ID:** af60a01a-12f7-4a83-94e8-ca8c78c87068<br>**Similarity:** 0.45218757<br>**Text:** ร้านค้า, สาขา, unit และสถานะว่ายังเช่าพื้นที่อยู่ หรือสิ้นสุดสัญญาไปแล้ว เพื่อตรวจสอบว่าเงินประกันยังคงมีอยู่หรือไม่อีกครั้ง

 

#การตรวจสอบเงินประกัน ต้องแยกวัตถุประสงค์การตรวจสอบไหม ว่าวัตถุประสงค์ลักษณะนี้ ต้องส่งตรวจสอบกับใคร

- สามารถส่งอีเมลถึง DL_TH-AR-INVOICE@lotuss.com ได้เลย ไม่ต้องแยกส่ง เพราะบัญชีที่ดูแลแต่ละ PN จะสามารถตรวจสอบ และรู้ที่มาที่ไปของเงินประกันแต่ละเจ้าได้ดีที่สุด<br>